In [4]:
import time
from pathlib import Path
from collections import Counter

import cv2
import numpy as np

# ── contour-finding config ───────────────────────────────────────────────────
RBC_HUE_MIN, RBC_HUE_MAX_WRAP = 130, 15
WBC_HUE_MIN, WBC_HUE_MAX = 120, 155
WBC_SAT_MIN = 40
WBC_REJECT_FRAC = 0.35
MIN_AREA, MAX_AREA, MIN_CIRCULARITY = 3500, 7500, 0.75
_CLOSE_KERNEL = np.ones((3, 3), np.uint8)

# ── size thresholds (real production classifier: livo_cell_features.py:587,
#    variables.py:198) -- hull area in px^2 ──────────────────────────────────
MICROCYTE_LO, MICROCYTE_HI = 1500, 5024
NORMAL_HI = 7350
MACROCYTE_HI = 10000

# ── color / pallor-scan config ───────────────────────────────────────────────
N_ANGLES = 36
R_MAX = 45
RIM_FRAC = 0.7          # pallor boundary = where saturation first reaches this fraction of the ray's rim level
HYPER_MAX = 0.15        # pallor_fraction < this -> Hyperchromic
HYPO_MIN = 0.33          # pallor_fraction > this -> Hypochromic (0.15-0.33 -> Normochromic)

_ANGLES = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
_COS = np.cos(_ANGLES)[:, None]
_SIN = np.sin(_ANGLES)[:, None]
_RS = np.arange(1, R_MAX + 1)[None, :]
_IDXS = np.arange(N_ANGLES)

print('config loaded')

config loaded


## 1. Contour finding

Otsu threshold on the HSV saturation channel, restricted to RBC hue, single `findContours` pass. Filtered by area + circularity (rejects touching/overlapping clusters) + WBC hue (rejects white cells). Returns raw `CHAIN_APPROX_NONE` points -- no smoothing, no simplification.

In [5]:
def foreground_mask(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, fg = cv2.threshold(hsv[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    hue = hsv[:, :, 0]
    rbc_hue = (hue >= RBC_HUE_MIN) | (hue <= RBC_HUE_MAX_WRAP)
    fg[~rbc_hue] = 0
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, _CLOSE_KERNEL)
    return fg, hsv


def isolated_cell_contours(img_bgr, min_area=MIN_AREA, max_area=MAX_AREA, min_circularity=MIN_CIRCULARITY):
    """Single findContours pass, filtered by area + circularity + WBC-color.
    Returns raw (N,2) float32 point arrays -- one per isolated RBC."""
    fg, hsv = foreground_mask(img_bgr)
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    out = []
    for c in contours:
        area = cv2.contourArea(c)
        perimeter = cv2.arcLength(c, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * area / (perimeter ** 2)
        if not (min_area <= area <= max_area and circularity >= min_circularity):
            continue

        bx, by, bw, bh = cv2.boundingRect(c)
        local_mask = np.zeros((bh, bw), np.uint8)
        cv2.drawContours(local_mask, [c - [bx, by]], -1, 255, -1)
        region = local_mask > 0
        hsv_crop = hsv[by:by + bh, bx:bx + bw]
        hue = hsv_crop[:, :, 0][region]
        sat = hsv_crop[:, :, 1][region]
        if hue.size == 0:
            continue
        wbc_frac = ((hue >= WBC_HUE_MIN) & (hue <= WBC_HUE_MAX) & (sat >= WBC_SAT_MIN)).mean()
        if wbc_frac >= WBC_REJECT_FRAC:
            continue

        out.append(c.reshape(-1, 2).astype(np.float32))
    return out

print('contour finding ready')

contour finding ready


## 2. Size

Hull area (not raw contour area -- robust to notches left over from separating touching cells).

In [6]:
def hull_area(pts):
    c = pts.reshape(-1, 1, 2).astype(np.float32)
    return cv2.contourArea(cv2.convexHull(c))


def size_label(area):
    if MICROCYTE_LO < area <= MICROCYTE_HI:
        return 'microcyte'
    if MICROCYTE_HI < area <= NORMAL_HI:
        return 'normal_size'
    if NORMAL_HI < area <= MACROCYTE_HI:
        return 'macrocyte'
    return 'unclassified_size'

print('size classification ready')

size classification ready


## 3. Color

Radial saturation-profile scan: from the cell's own centroid, sample saturation along 36 rays out to the hull boundary. Per ray, the pallor boundary is where the value first rises to `RIM_FRAC` of that ray's own rim level. Pallor fraction = mean(boundary radius / ray length) across all rays. No hard blob threshold on the whole cell -- avoids the two failure modes already tried and rejected (adaptive-threshold hierarchy tracing the whole cell; percentile-of-image color splitting on noise).

Vectorized across all 36 rays at once per cell (~0.12ms/cell).

In [7]:
def pallor_fraction(pts, sat_full):
    H, W = sat_full.shape
    c = pts.reshape(-1, 1, 2).astype(np.float32)
    hull = cv2.convexHull(c)
    M = cv2.moments(hull)
    cx, cy = M['m10'] / M['m00'], M['m01'] / M['m00']

    xs = cx + _COS * _RS
    ys = cy + _SIN * _RS
    xs_i = np.clip(xs, 0, W - 1).astype(np.int32)
    ys_i = np.clip(ys, 0, H - 1).astype(np.int32)

    bx, by, bw, bh = cv2.boundingRect(hull.astype(np.int32))
    pad = 3
    ox, oy = bx - pad, by - pad
    mw, mh = bw + 2 * pad, bh + 2 * pad
    local_mask = np.zeros((mh, mw), np.uint8)
    cv2.drawContours(local_mask, [(hull.reshape(-1, 2) - [ox, oy]).astype(np.int32)], -1, 255, -1)
    lxs = np.clip(xs_i - ox, 0, mw - 1)
    lys = np.clip(ys_i - oy, 0, mh - 1)
    inside = local_mask[lys, lxs] > 0

    first_false = np.argmax(~inside, axis=1)
    all_inside = inside.all(axis=1)
    r_max_idx = np.where(all_inside, R_MAX - 1, np.maximum(first_false - 1, 0))

    sat_vals = sat_full[ys_i, xs_i]
    rim_idx = np.clip(r_max_idx[:, None] - np.array([2, 1, 0]), 0, R_MAX - 1)
    rim_level = sat_vals[_IDXS[:, None], rim_idx].mean(axis=1)
    thresh = rim_level * RIM_FRAC

    cross = sat_vals >= thresh[:, None]
    valid_range = np.arange(R_MAX)[None, :] <= r_max_idx[:, None]
    cross = cross & valid_range
    any_cross = cross.any(axis=1)
    first_cross_idx = np.argmax(cross, axis=1)
    pallor_r = np.where(any_cross, _RS[0, first_cross_idx], _RS[0, r_max_idx]).astype(np.float32)

    r_max_val = _RS[0, r_max_idx]
    valid = r_max_val >= 4
    if not valid.any():
        return 0.0
    return float((pallor_r[valid] / np.maximum(r_max_val[valid], 1)).mean())


def color_label(frac):
    if frac < HYPER_MAX:
        return 'Hyperchromic'
    if frac <= HYPO_MIN:
        return 'Normochromic'
    return 'Hypochromic'

print('color classification ready')

color classification ready


## Run on one image

In [10]:
IMG_PATH = 'F:/Livo/Data - 2026/Rbc/others/7/Img_7_13.jpg'
OUT_DIR = Path('F:/Livo/Data - 2026/Rbc/traditional-cv/notebook_output')
OUT_DIR.mkdir(exist_ok=True)

SIZE_COLOR = {'microcyte': (255, 140, 0), 'normal_size': (0, 200, 0), 'macrocyte': (0, 0, 255), 'unclassified_size': (150, 150, 150)}
COLOR_TAG = {'Hyperchromic': 'Hyper', 'Normochromic': 'Norm', 'Hypochromic': 'Hypo'}

img = cv2.imread(IMG_PATH)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
sat_full = hsv[:, :, 1].astype(np.float32)

t0 = time.perf_counter()
contours = isolated_cell_contours(img)
t_contour = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
sizes = [size_label(hull_area(pts)) for pts in contours]
t_size = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
fracs = [pallor_fraction(pts, sat_full) for pts in contours]
t_color = (time.perf_counter() - t0) * 1000
cols = [color_label(f) for f in fracs]

results = list(zip(contours, sizes, fracs, cols))

vis = img.copy()
for pts, sz, frac, col in results:
    poly = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(vis, [poly], True, SIZE_COLOR[sz], 1, cv2.LINE_AA)
    bx, by, bw, bh = cv2.boundingRect(poly)
    cv2.putText(vis, f'{sz[:5]}/{COLOR_TAG[col]}', (bx, by - 2), cv2.FONT_HERSHEY_PLAIN, 0.6, (255, 0, 255), 1, cv2.LINE_AA)
cv2.imwrite(str(OUT_DIR / Path(IMG_PATH).name), vis)

print(f'{len(contours)} cells')
print(f'  contour: {t_contour:.2f} ms  ({t_contour/max(len(contours),1):.4f} ms/cell)')
print(f'  size:    {t_size:.2f} ms  ({t_size/max(len(contours),1):.4f} ms/cell)')
print(f'  color:   {t_color:.2f} ms  ({t_color/max(len(contours),1):.4f} ms/cell)')
print(f'  total:   {t_contour+t_size+t_color:.2f} ms')
print('size:', dict(Counter(sizes)))
print('color:', dict(Counter(cols)))
print('saved ->', OUT_DIR / Path(IMG_PATH).name)


93 cells
  contour: 19.17 ms  (0.2061 ms/cell)
  size:    2.22 ms  (0.0239 ms/cell)
  color:   13.58 ms  (0.1460 ms/cell)
  total:   34.97 ms
size: {'microcyte': 68, 'normal_size': 25}
color: {'Normochromic': 62, 'Hypochromic': 26, 'Hyperchromic': 5}
saved -> F:\Livo\Data - 2026\Rbc\traditional-cv\notebook_output\Img_7_13.jpg


## Run on a batch of images

In [11]:
BATCH_DIR = Path('F:/Livo/Data - 2026/Rbc/others/7')
BATCH_NAMES = [f'Img_0_{i}.jpg' for i in range(10)]

total_cells = 0
total_t_contour, total_t_size, total_t_color = 0.0, 0.0, 0.0
all_sizes, all_colors = [], []

for name in BATCH_NAMES:
    img = cv2.imread(str(BATCH_DIR / name))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    sat_full = hsv[:, :, 1].astype(np.float32)

    t0 = time.perf_counter()
    contours = isolated_cell_contours(img)
    total_t_contour += (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    sizes = [size_label(hull_area(pts)) for pts in contours]
    total_t_size += (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    fracs = [pallor_fraction(pts, sat_full) for pts in contours]
    total_t_color += (time.perf_counter() - t0) * 1000
    cols = [color_label(f) for f in fracs]

    all_sizes.extend(sizes); all_colors.extend(cols)
    total_cells += len(contours)

    vis = img.copy()
    for pts, sz, col in zip(contours, sizes, cols):
        poly = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(vis, [poly], True, SIZE_COLOR[sz], 1, cv2.LINE_AA)
        bx, by, bw, bh = cv2.boundingRect(poly)
        cv2.putText(vis, f'{sz[:5]}/{COLOR_TAG[col]}', (bx, by - 2), cv2.FONT_HERSHEY_PLAIN, 0.6, (255, 0, 255), 1, cv2.LINE_AA)
    cv2.imwrite(str(OUT_DIR / name), vis)

n_img = len(BATCH_NAMES)
print(f'{total_cells} cells across {n_img} images')
print(f'  contour: {total_t_contour:.2f} ms total  ({total_t_contour/n_img:.2f} ms/image avg,  {total_t_contour/total_cells:.4f} ms/cell)')
print(f'  size:    {total_t_size:.2f} ms total  ({total_t_size/n_img:.2f} ms/image avg,  {total_t_size/total_cells:.4f} ms/cell)')
print(f'  color:   {total_t_color:.2f} ms total  ({total_t_color/n_img:.2f} ms/image avg,  {total_t_color/total_cells:.4f} ms/cell)')
total_all = total_t_contour + total_t_size + total_t_color
print(f'  TOTAL:   {total_all:.2f} ms  ({total_all/n_img:.2f} ms/image avg)')
print('size:', dict(Counter(all_sizes)))
print('color:', dict(Counter(all_colors)))
print('saved ->', OUT_DIR)


1007 cells across 10 images
  contour: 200.51 ms total  (20.05 ms/image avg,  0.1991 ms/cell)
  size:    17.15 ms total  (1.71 ms/image avg,  0.0170 ms/cell)
  color:   164.67 ms total  (16.47 ms/image avg,  0.1635 ms/cell)
  TOTAL:   382.33 ms  (38.23 ms/image avg)
size: {'microcyte': 799, 'normal_size': 201, 'macrocyte': 7}
color: {'Normochromic': 545, 'Hypochromic': 310, 'Hyperchromic': 152}
saved -> F:\Livo\Data - 2026\Rbc\traditional-cv\notebook_output
